# Stage 1 — Cohort Selection

Select MIMIC-IV patients with ≥2 admissions (ICD-10 + discharge notes).

**Patient-centric model:**
- **Latest admission** → note with **discharge package redacted** (diagnosis, instructions, meds, disposition, condition, followup, transitional issues) + Hospital Course `# Problem:` titles scrubbed
- **Ground truth ICD-10** stored separately (never sent to LLM)
- **Prior admissions** → detailed history (HPI, hospital course, prior discharge diagnoses)

Run **`00_settings.ipynb`** first.

**Output:** `data/cohort/cohort.pkl`

**Next:** `stage_02_information_extraction.ipynb`


## Setup

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if (ROOT / "pipeline.py").exists():
    NB_DIR = ROOT
elif (ROOT / "notebooks" / "pipeline.py").exists():
    NB_DIR = ROOT / "notebooks"
else:
    NB_DIR = ROOT.parent / "notebooks"
sys.path.insert(0, str(NB_DIR))

import pandas as pd
from pipeline import (
    COHORT_INDEX_JSON,
    COHORT_PICKLE,
    MIN_ADMISSIONS_PER_PATIENT,
    MIN_NOTE_CHARS,
    N_PATIENTS,
    PHYSIONET_ROOT,
    RANDOM_SEED,
    load_cohort_index,
    load_mimic_cohort,
    save_cohort,
)

pd.set_option("display.max_colwidth", 120)
print(f"MIMIC root: {PHYSIONET_ROOT}")
print(f"Target patients: {N_PATIENTS} | seed: {RANDOM_SEED}")


MIMIC root: /Users/narenkhatwani/Desktop/physionet.org/files
Target patients: 15 | seed: 42


## Load & Sample Cohort

In [2]:
cohort_df = load_mimic_cohort(
    n_patients=N_PATIENTS,
    min_admissions=MIN_ADMISSIONS_PER_PATIENT,
    min_note_chars=MIN_NOTE_CHARS,
    seed=RANDOM_SEED,
)

print(f"Patients (1 row each): {len(cohort_df)}")
print(f"Avg prior admissions as history: {cohort_df['n_prior_admissions'].mean():.1f}")
cohort_df[[
    "patient_id", "admission_id", "n_prior_admissions", "n_total_admissions",
    "gender", "anchor_age", "admittime", "primary_icd_code", "primary_dx_title", "text_len",
]].head(10)


Loading discharge notes (this may take ~20s)...
Patients: 15
Admissions: 50


,patient_id,admission_id,gender,anchor_age,admittime,admission_type,primary_icd_code,primary_dx_title,text_len
0,10361982,28540049,F,34,2162-05-08 09:30:00,DIRECT OBSERVATION,N939,"Abnormal uterine and vaginal bleeding, unspecified",6696
1,10361982,24286431,F,34,2162-05-25 16:55:00,DIRECT OBSERVATION,N99820,Postprocedural hemorrhage of a genitourinary system organ or structure following a genitourinary system procedure,5497
2,10426859,25245894,F,91,2193-12-20 17:16:00,DIRECT OBSERVATION,R0789,Other chest pain,13278
3,10426859,21013288,F,91,2194-04-30 16:41:00,OBSERVATION ADMIT,I471,Supraventricular tachycardia,10713
4,10426859,29908281,F,91,2194-05-21 23:23:00,OBSERVATION ADMIT,I471,Supraventricular tachycardia,11361
5,10458324,21719653,M,57,2176-11-17 01:31:00,EU OBSERVATION,J09X2,Influenza due to identified novel influenza A virus with other respiratory manifestations,11044
6,10458324,21744342,M,57,2178-07-14 18:25:00,OBSERVATION ADMIT,S270XXA,"Traumatic pneumothorax, initial encounter",12929
7,11251337,27937520,M,55,2164-10-11 23:35:00,OBSERVATION ADMIT,N132,Hydronephrosis with renal and ureteral calculous obstruction,10395
8,11251337,21161726,M,55,2164-11-29 13:57:00,OBSERVATION ADMIT,N136,Pyonephrosis,10164
9,11251337,29568708,M,55,2164-12-23 09:38:00,DIRECT OBSERVATION,N132,Hydronephrosis with renal and ureteral calculous obstruction,7627


## Patient Summary

In [3]:
cohort_df[[
    "patient_id", "n_total_admissions", "n_prior_admissions",
    "admittime", "primary_icd_code", "primary_dx_title",
]].sort_values("n_prior_admissions", ascending=False)


,patient_id,admissions,admission_types,primary_dx
0,10361982,2,DIRECT OBSERVATION,"Abnormal uterine and vaginal bleeding, unspecified | Postprocedural hemorrhage of a genitourinary system organ or st..."
1,10426859,3,"DIRECT OBSERVATION, OBSERVATION ADMIT",Other chest pain | Supraventricular tachycardia
2,10458324,2,"EU OBSERVATION, OBSERVATION ADMIT","Influenza due to identified novel influenza A virus with other respiratory manifestations | Traumatic pneumothorax, ..."
3,11251337,3,"DIRECT OBSERVATION, OBSERVATION ADMIT",Hydronephrosis with renal and ureteral calculous obstruction | Pyonephrosis
4,11474876,2,"AMBULATORY OBSERVATION, OBSERVATION ADMIT","Cholangitis | Calculus of bile duct with cholangitis, unspecified, without obstruction"
5,11607177,5,"DIRECT EMER., ELECTIVE, EU OBSERVATION, EW EMER.","Umbilical hernia with obstruction, without gangrene | Umbilical hernia without obstruction or gangrene | Postprocedu..."
6,12007928,3,EW EMER.,Syncope and collapse | Acute on chronic diastolic (congestive) heart failure
7,13196707,2,OBSERVATION ADMIT,"Malignant (primary) neoplasm, unspecified | Sepsis, unspecified organism"
8,13508515,2,"EW EMER., SURGICAL SAME DAY ADMISSION","Acute on chronic systolic (congestive) heart failure | Malignant neoplasm of left kidney, except renal pelvis"
9,13952483,3,"DIRECT EMER., EW EMER.","Malignant neoplasm of lower lobe, left bronchus or lung | Pneumonia due to other Gram-negative bacteria | Other hypo..."


## Preview One Note

In [4]:
from pipeline import format_admission_history_text

sample = cohort_df.iloc[0]
history = sample.get('admission_history') or []
print(f"patient_id={sample['patient_id']} | latest hadm_id={sample['hadm_id']}")
print(f"Prior admissions: {len(history)}")
print(f"Ground truth primary: {sample['primary_icd_code']} — {sample['primary_dx_title']}")
red = sample.get('redacted_diagnosis_text') or ''
print(f"Redacted discharge-package chars: {len(red)} (0 = package past truncate or absent)")
print('=' * 60)
print('CODING NOTE (redacted, first 700 chars):')
print(sample['clinical_note'][:700], '...')
if history:
    print('\n' + '=' * 60)
    print('PRIOR HISTORY (detailed):')
    print(format_admission_history_text(history)[:2000])


subject_id=10361982 hadm_id=28540049
Primary ICD-10: N939 — Abnormal uterine and vaginal bleeding, unspecified
All codes (11): ['N939', 'N800', 'D261', 'N838', 'Z23']...
------------------------------------------------------------
 
Name:  ___                    Unit No:   ___
 
Admission Date:  ___              Discharge Date:   ___
 
Date of Birth:  ___             Sex:   F
 
Service: OBSTETRICS/GYNECOLOGY
 
Allergies: 
Compazine
 
Attending: ___
 
Chief Complaint:
menorrhagia
 
Major Surgical or Invasive Procedure:
total laparoscopic hysterectomy, bilateral salpingectomy, 
cystoscopy

 
History of Present Illness:
___ yo G6 ___ LMP ___ with a history of abnormal uterine
bleeding. She cycles approximately every 30 days and bleeding
heavily for ___ weeks. During her heaviest bleeding period, she
changes a pad/tampon every 15 minutes. She endorses 
dysmenorrhea,
intra-menstrual bleeding and post-coital bleeding. She denies
dyspareunia. She takes NSAIDs daily without total resolution of

## Save Cohort (for Stage 2+)

In [5]:
saved_path = save_cohort(cohort_df, seed=RANDOM_SEED)
index = load_cohort_index()

print(f"Saved cohort → {saved_path}")
print(f"Index       → {COHORT_INDEX_JSON}")
print(f"Patients    → {index['n_patients']}")
print(f"Rows saved  → {index.get('n_index_admissions', len(cohort_df))} (1 per patient, latest note)")
print("Subject IDs:", index["subject_ids"])


Saved cohort → /Users/narenkhatwani/GitHub/ai-agents-for-clinical-coding/data/cohort/cohort.pkl
Index       → /Users/narenkhatwani/GitHub/ai-agents-for-clinical-coding/data/cohort/cohort_index.json
Patients    → 15
Admissions  → 50
Subject IDs: [10361982, 10426859, 10458324, 11251337, 11474876, 11607177, 12007928, 13196707, 13508515, 13952483, 16014068, 17774110, 18412100, 19104262, 19632936]
